In [0]:
import sys
import importlib
sys.path.append("/Workspace/Shared/Capstone_final/Databricks/")

import table_config
import bronze_engine
import utils
import checkpoint_manager
importlib.reload(checkpoint_manager)
importlib.reload(utils)
importlib.reload(table_config)
importlib.reload(bronze_engine)

from bronze_engine import BronzeEngine
from table_config import REFERENCE_TABLES, TRANSACTIONAL_TABLES

try:
    batch_number = dbutils.widgets.get("batch_number")
except:
    dbutils.widgets.text("batch_number", "1")
    batch_number = "1"

print(f"Starting Bronze for batch: {batch_number}")
BronzeEngine(spark).run(batch_number)

In [0]:
# In a notebook cell
dbutils.fs.rm("s3://capstone-ecomm-team8/delta/bronze/order_reviews", True)

from checkpoint_manager import CheckpointManager
CheckpointManager(spark).reset("1", "bronze")
print("Ready")

In [0]:
# ============================================================================
# COMPLETE BRONZE REBUILD SCRIPT
# ============================================================================

from bronze_engine import BronzeEngine
from table_config import S3_DELTA_BRONZE
import importlib

importlib.reload(checkpoint_manager)
from checkpoint_manager import CheckpointManager

print("=" * 80)
print("REBUILDING BRONZE LAYER WITH CONSISTENT SCHEMA")
print("=" * 80)

# Step 1: Delete ALL existing Bronze Delta tables
print("\n[1/4] Deleting all existing Bronze tables...")
bronze_tables = [
    "category_translation", "geolocation", "sellers", "customers", "products",
    "orders", "order_items", "order_payments", "order_reviews"
]

for table in bronze_tables:
    s3_path = f"{S3_DELTA_BRONZE}/{table}"
    try:
        dbutils.fs.rm(s3_path, True)
        print(f"  ✓ Deleted {table}")
    except Exception as e:
        print(f"  ⚠️ {table} not found or already deleted (OK)")

# Step 2: Drop catalog entries
print("\n[2/4] Dropping catalog tables...")
for table in bronze_tables:
    try:
        spark.sql(f"DROP TABLE IF EXISTS bronze.{table}")
        print(f"  ✓ Dropped bronze.{table}")
    except Exception as e:
        print(f"  ⚠️ Could not drop bronze.{table} (OK)")

# Step 3: Clear all checkpoints
print("\n[3/4] Clearing checkpoints...")
cp = CheckpointManager(spark)

for batch in ["1", "2", "3", "4"]:
    cp.clear(batch, "bronze")
    cp.clear(batch, "silver")

# Step 4: Re-ingest with new schema-consistent code
print("\n[4/4] Re-ingesting all batches with consistent schema...")

bronze_engine = BronzeEngine(spark)

# Batch 1 (includes reference tables)
print("\n" + "=" * 80)
print("BATCH 1")
print("=" * 80)
bronze_engine.run("1")

# Batch 2
print("\n" + "=" * 80)
print("BATCH 2")
print("=" * 80)
bronze_engine.run("2")

# Uncomment if you have batches 3 and 4
# print("\n" + "=" * 80)
# print("BATCH 3")
# print("=" * 80)
# bronze_engine.run("3")

# print("\n" + "=" * 80)
# print("BATCH 4")
# print("=" * 80)
# bronze_engine.run("4")

print("\n" + "=" * 80)
print("BRONZE REBUILD COMPLETE")
print("=" * 80)

# Step 5: Verify schemas are consistent
print("\n[VERIFICATION] Checking schema consistency...")

for table in ["orders", "order_items", "order_payments", "order_reviews"]:
    try:
        df = spark.read.format("delta").load(f"{S3_DELTA_BRONZE}/{table}")
        print(f"\n{table}:")
        print(f"  Row count: {df.count():,}")
        print(f"  Schema:")
        for field in df.schema.fields:
            if field.name not in ["ingestion_timestamp", "source_file", "batch_id"]:
                print(f"    - {field.name}: {field.dataType}")
    except Exception as e:
        print(f"\n{table}: ERROR - {e}")

# Step 6: Show distinct values for key columns
print("\n[VALIDATION] Checking data quality...")

try:
    df = spark.read.format("delta").load(f"{S3_DELTA_BRONZE}/order_reviews")
    print("\norder_reviews.review_score distinct values:")
    df.select("review_score").distinct().orderBy("review_score").show()
except Exception as e:
    print(f"Could not validate order_reviews: {e}")

try:
    df = spark.read.format("delta").load(f"{S3_DELTA_BRONZE}/order_items")
    print("\norder_items.order_item_id sample values:")
    df.select("order_item_id").limit(10).show()
except Exception as e:
    print(f"Could not validate order_items: {e}")

print("\n" + "=" * 80)
print("ALL DONE!")
print("=" * 80)

In [0]:
# ============================================================================
# MANUAL CHECKPOINT DELETION + REBUILD
# After you manually delete checkpoint files in S3 console
# ============================================================================

from bronze_engine import BronzeEngine
from silver_engine import SilverEngine
from gold_engine import GoldEngine

# Make sure you've manually deleted these in S3:
# - s3://capstone-ecomm-team8/checkpoints/batch_1/bronze.json
# - s3://capstone-ecomm-team8/checkpoints/batch_1/silver.json
# - s3://capstone-ecomm-team8/checkpoints/batch_1/gold.json (if exists)

print("Running full pipeline for Batch 1...")

# Bronze
bronze_engine = BronzeEngine(spark)
bronze_engine.run("1")

# Silver
silver_engine = SilverEngine(spark)
silver_engine.run("1")

# Gold
gold_engine = GoldEngine(spark)
gold_engine.run("1")

print("Done! Verify your schemas are correct.")

In [0]:
# Test if read_csv is actually applying types
from table_config import TRANSACTIONAL_TABLES
import importlib
importlib.reload(utils)
import utils

test_df = utils.read_csv(
    spark, 
    "s3://capstone-ecomm-team8/raw/batch_1/order_reviews_dataset.csv",
    TRANSACTIONAL_TABLES["order_reviews"]
)

print("Schema after read_csv:")
test_df.printSchema()

print("\nSample data:")
test_df.select("review_score", "review_creation_date").show(5, truncate=False)

In [0]:
# ============================================================================
# INLINE FIX - Define read_csv directly in your notebook
# ============================================================================

from pyspark.sql.functions import to_timestamp, col

def read_csv_fixed(spark, s3_path, table_config=None):
    """
    Read CSV with headers and schema enforcement
    
    CRITICAL: Read as strings first, then apply proper types
    """
    # Step 1: Always read as strings first - NO SCHEMA INFERENCE
    df = spark.read.option("header", True).option("inferSchema", False).csv(s3_path)
    
    print(f"DEBUG: Initial schema - all strings")
    
    # Step 2: Apply schema from config if provided
    if table_config and "schema" in table_config:
        schema_dict = table_config["schema"]
        print(f"DEBUG: Applying schema from config...")
        
        for col_name, dtype in schema_dict.items():
            if col_name in df.columns:
                dtype_lower = str(dtype).lower()
                
                # Apply type casting based on config
                if dtype_lower == "timestamp":
                    print(f"  → {col_name}: string → timestamp")
                    df = df.withColumn(col_name, to_timestamp(col(col_name)))
                elif dtype_lower in ["int", "integer"]:
                    print(f"  → {col_name}: string → int")
                    df = df.withColumn(col_name, col(col_name).cast("int"))
                elif dtype_lower in ["double", "float"]:
                    print(f"  → {col_name}: string → double")
                    df = df.withColumn(col_name, col(col_name).cast("double"))
                elif dtype_lower != "string":
                    print(f"  → {col_name}: string → {dtype_lower}")
                    df = df.withColumn(col_name, col(col_name).cast(dtype))
    else:
        print("DEBUG: No table_config provided, returning all strings")
    
    return df

# Test the inline version
from table_config import TRANSACTIONAL_TABLES

print("Testing inline read_csv_fixed function:")
print("=" * 80)

test_df = read_csv_fixed(
    spark, 
    "s3://capstone-ecomm-team8/raw/batch_1/order_reviews_dataset.csv",
    TRANSACTIONAL_TABLES["order_reviews"]
)

print("\n" + "=" * 80)
print("Final Schema:")
test_df.printSchema()

print("\nSample data:")
test_df.select("review_score", "review_creation_date").show(5, truncate=False)

In [0]:
# Check what's actually in the config
from table_config import TRANSACTIONAL_TABLES

print("order_reviews config:")
print(TRANSACTIONAL_TABLES.get("order_reviews"))

print("\nDoes it have 'schema' key?")
print("schema" in TRANSACTIONAL_TABLES.get("order_reviews", {}))

print("\nWhat's in the schema?")
print(TRANSACTIONAL_TABLES.get("order_reviews", {}).get("schema"))

In [0]:
import importlib
import table_config
importlib.reload(table_config)
from table_config import TRANSACTIONAL_TABLES

for table_name, config in TRANSACTIONAL_TABLES.items():
    has_schema = "schema" in config
    print(f"{table_name}: has 'schema' = {has_schema}")
    if has_schema:
        print(f"  Schema keys: {list(config['schema'].keys())}")
    else:
        print(f"  ⚠️ MISSING SCHEMA!")